**Tabla de contenido**

- [Introducción](#Introduccio)
- [Librerías](#Librerias)
- [Preprocesamiento](#Preprocesamiento)


# Introduccion

Tu tarea consiste en crear un clasificador binario que prediga si un comentario de Reddit infringe una norma específica. El conjunto de datos procede de una gran colección de comentarios moderados, con una serie de normas de subreddit, tonos y expectativas de la comunidad.

`dataset`
- **body** - el texto del comentario
- **rule** - la regla que se considera que infringe el comentario
- **subreddit** - el foro en el que se hizo el comentario
- **positive_example_{1,2}** - ejemplos de comentarios que infringen la regla
- **negative_example_{1,2}** - ejemplos de comentarios que no infringen la regla
- **rule_violation** - el objetivo binario


# Librerias

In [ ]:
import os
import pandas as pd
import re

In [147]:
file_path = lambda file: os.path.join(os.getcwd(),'data/Agile Community Rules Classification',file)
train = pd.read_csv(file_path('train.csv'))
#train = train.set_index('row_id', drop=True)
#pd.set_option('display.max_colwidth', None)  # Mostrar todo el contenido de las celd
train.head(2)

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
0,0,Banks don't want you to know this! Click here to know more!,"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",Futurology,"If you could tell your younger self something different about sex, what would that be?\n\ni AM IN A CONTEST TO WIN FUNDING FOR MY SEX POSITIVE FILM: VOTE HERE:\n\nhttp://sheknows.offerpop.com/campaign/813112/entry/v144417",hunt for lady for jack off in neighbourhood http://url.inmusi.com/gakq,Watch Golden Globe Awards 2017 Live Online in HD Coverage without ADS (VIP STREAMS)\n=\n\nHD STREAM QUALITY >>> [WATCH LINK1](http://forum.submitexpress.com/viewtopic.php?f=9&t=215858)\n=\n\nHD BROADCASTING QUALITY >>> [WATCH LINK1](http://forum.submitexpress.com/viewtopic.php?f=9&t=215858)\n=\n\nMobile Compatibility: YES\n=\n\nNO ADS | NO ADS | ADS\n=\n,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/STREAM:\n\nhttp://music.theblacksmithed.com/download/birds/",0
1,1,SD Stream [ ENG Link 1] (http://www.sportsstreams247.com/astra-giurgiu-vs-fc-austria-wien/),"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",soccerstreams,[I wanna kiss you all over! Stunning!](http://www.oilflush.life/2017/01/26/6/),"LOLGA.COM is One of the First Professional Online Gold sites. By Now, As A Game Gold Seller, we've over more than 5 yrs Of Experience And Can Pass That On To Our Customers.","#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTube Search Beanie 864 Click Link BELOW To Hear Hit Single\n ""Ah Man"" \n Beanie 864 FEAT King Kota \n (King Kota Is Only 15!) Lit 🌡🔥👍💵💯Fr Fr \nhttps://youtu.be/tLqbV1Jmt5Y","[15 Amazing Hidden Features Of Google Search You Probably Don’t Know](http://www.madpeoples.com/2017/01/02 No one would argue the fact that Google is one of the most useful sihttp://www.madpeoples.com/2016/12/31/15-amazing-hidden-features-of-google-search-you-probably-dont-know/tes on the Internet. Unfortunately, most people only use about...?utm_source=reddit&utm_campaign=samreen&utm_medium=cpc)",0


In [ ]:
print(train['positive_example_1'][4])

# Preprocesamiento

Vamos a prepar los datos para el modelo Bertweet. BERTweet es un modelo de lenguaje basado en la arquitectura BERT (Bidirectional Encoder Representations from Transformers), pero específicamente entrenado en tweets (textos de Twitter) en ingles. Está optimizado para lenguaje informal, lo que incluye jerga de redes sociales, hashtags, emoticonos, menciones (@) y ortografía no estándar (ej: "loooove"). Tiene un Tokenizador adaptado: Maneja mejor palabras repetidas ("goooool"), contracciones ("don't" → "do n't") y palabras concatenadas ("NewYork").

Este modelo está diseñado para tareas de Procesamiento de Lenguaje Natural (NLP) en redes sociales, como:

1. `Clasificación de Texto`

- Análisis de sentimiento (ej.: ¿Es un tweet positivo, negativo o neutro?).
- Detección de hate speech, spam o bullying.
- Identificación de noticias falsas (fake news) en redes sociales.

2. `Extracción de Información`

- Named Entity Recognition (NER): Identificar personas, lugares, etc., en tweets.
- Detección de temas (topic modeling) en conversaciones de Twitter.

3. `Aplicaciones Específicas`

- Moderación automática de contenido en plataformas sociales.
- Respuesta a preguntas (QA) en contextos informales.
- Generación de texto (aunque no es su enfoque principal).

Esto implica que el preprocesamiento de los textos debe realizarse de la siguiente forma:

1. Reemplazar las URL por la abreviatura `[URL]`.
2. Reemplazar las mensiones de usarios por la abreviatura `[USER]`
3. Los `Hashtag` deben dejarse tal cual como están.
4. Los emojis deben dejarse ya que ayudan al modelo a entender tono emosional o sarcasmo.
5. Se deben eliminar múltiples espacios, tabs o saltos de linea innecesarios.
6. No convertir a mayúscula o minúscula, ni eliminar los signos de puntuación. Este modelo no distigue entre mayúscula/minúscula.

In [162]:
from urlextract import URLExtract

def cleantext_toBERTweet(text):
    text = re.sub(r"\s+"," ", text).strip()                     # reemplaza múltiples espacios, tabs o saltos de línea por un solo espacio
    email_pattern = r'\b([A-Za-z0-9._%+-]+)\s*(?:@|\[at\]|\(at\)|arroba)\s*([A-Za-z0-9.-]+)\s*(?:\.|\[dot\]|\(dot\)|punto)\s*([A-Za-z]{2,})\b'
    text = re.sub(email_pattern,'[EMAIL]',text)                 # reemplaza correo electrónicos a [EMAIL]
    phone_pattern = r'(?<!\w)(?:\+?\d{1,3}|\(\+?\d{1,3}\))?(?:[-. /]?\d{2,4}){2,5}(?:[-. /]?\d{2,})\b(?:[ ]*(?:ext|xtn|x|#)[ ]*\d{1,6})?(?!\w)'
    text = re.sub(phone_pattern, '[PHONE]', text)               # Reemplaza números de teléfonos por [PHONE]

    # Reemplazo de URLs estándar detectadas por URLExtract
    extractor = URLExtract()
    urls = extractor.find_urls(text)
    for url in urls:
        text = text.replace(url, '[URL]')  
    text = re.sub(r"@\w+", " [USER] ", text)                    # reemplaza usuarios por [USER]

    # Patrón “raro” en url (/p/... .xxx)
    pattern_url_raro =  r"/[A-Za-z]/[\w-]+\.[A-Za-z]{2,4}\b"
    text = re.sub(pattern_url_raro, "[URL]", text)
    # Cualquier HTTP/HTTPS
    pattern_any_url = r"(?:https?://|://)[^\s]+"
    text = re.sub(pattern_any_url, "[URL]", text)
    # patrones raros
    pattern_scheme = r"\b[a-z][\w+.-]*://[^\s]+\b"
    text = re.sub(pattern_scheme, "[URL]", text)

    # “come” todas las aperturas de paréntesis o corchetes adyacentes antes de un token del tipo […]
    pattern =  r'([(\[])\[URL\]([)\]])'  # Captura ( [URL] ) o [ [URL] ]
    text = re.sub(pattern,'[URL]',text)
    text = re.sub(r"\*", "", text)
    text = re.sub(r'\/','',text)
    return text

Veamos ahora como quedan los texto, para esto sacaremos muestras aleatorias y las limpiaremos, esto con el fin de saber que todo está ok.

In [193]:
#train = train.set_index('row_id', drop=True)
pd.set_option('display.max_colwidth', None)  # Mostrar todo el contenido de las celd
muestra = train['negative_example_1'].sample(n=10)
muestra.head(10)

1717                                                                                                                                                                                                                                                                                                                              [Full Vid](http://hegytr.com/YuzL0) She's Chloe Lamb
946                                                                                                                                                                                                                                                                                                                        You didn't sexually assault her, it's just morning regret. 
982                                                                                                                                                                                                                                                       

In [194]:
muestra = muestra.apply(cleantext_toBERTweet)
muestra.head(10)

1717                                                                                                                                                                                                                                                                                         [Full Vid][URL] She's Chloe Lamb
946                                                                                                                                                                                                                                                                You didn't sexually assault her, it's just morning regret.
982                                                                                                                                                                                                                                                             [This Raw Food Cut Into Perfect Cubes Is Oddly Soothing][URL]
319                                           

In [163]:
texto = "inspiration? Thats putting it mildly :-) ://youtu.be/lQ_OIZhFHeA"

In [164]:
texto = str(texto)
texto = cleantext_toBERTweet(texto)
print(texto)

inspiration? Thats putting it mildly :-) [URL]


In [139]:
texto = "SD // [URL] // Lang: nl"
patron = r'\/'
texto_limpio = re.sub(patron, '', texto)

print(texto_limpio)

SD  [URL]  Lang: nl
